# Lab 09: Dashboard de Analytics com Agentes Inteligentes (UI)

**Disciplina:** Augmented Analytics & AI-Driven Insights
**Objetivo:** Criar uma interface (Dashboard) para a Squad de Agentes da Aula 03 utilizando Streamlit.

---
### 🚀 1. Instalação de Dependências

In [ ]:
!pip install crewai streamlit pyngrok pandas -q

### 📝 2. Desenvolvimento do Dashboard (app.py)
A célula abaixo criará o arquivo `app.py` no ambiente do Colab.

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import os
from crewai import Agent, Task, Crew, Process

# -----------------------------------------------------------------
# 1. CONFIGURAÇÃO DA UI
# -----------------------------------------------------------------
st.set_page_config(page_title="BI Agentic Dashboard", layout="wide")

st.title("🤖 Agentic Analytics Dashboard")
st.subheader("Transformando Dados em Insights com Squads de IA")

with st.sidebar:
    st.header("⚙️ Configurações")
    api_key = st.text_input("Insira sua Gemini API Key:", type="password")
    model_choice = st.selectbox("Selecione o Modelo:", ["gemini/gemini-3.8-flash"])
    st.divider()
    st.info("MBA em BI & Analytics - FIAP")

# -----------------------------------------------------------------
# 2. UPLOAD DE DADOS
# -----------------------------------------------------------------
uploaded_file = st.file_uploader("Faça o upload do seu dataset (CSV):", type="csv")

if uploaded_file and api_key:
    df = pd.read_csv(uploaded_file)
    
    # Cards Executivos de BI
    col1, col2, col3 = st.columns(3)
    col1.metric("📊 Total de Registros", len(df))
    col2.metric("📋 Total de Colunas", len(df.columns))
    col3.metric("🔢 Colunas Numéricas", len(df.select_dtypes(include='number').columns))
    
    st.write("### 🔍 Amostra dos Dados Carregados", df.head(5))
    
    # Disparo com persistência em session_state
    if st.button("🚀 Disparar Squad de Agentes"):
        os.environ["GEMINI_API_KEY"] = api_key
        os.environ["GOOGLE_API_KEY"] = api_key
        
        with st.spinner("🤖 Agentes analisando dados... Aguarde."):
            analyst = Agent(
                role='Analista de Dados Sênior',
                goal='Identificar padrões críticos nos dados fornecidos',
                backstory='Especialista em estatística e comportamento do consumidor.',
                llm=model_choice
            )
            
            task = Task(
                description=f'Analise as variáveis {df.columns.tolist()} na amostra a seguir e apresente um sumário executivo: {df.head(10).to_markdown()}',
                expected_output='Um relatório Markdown com 3 insights estratégicos.',
                agent=analyst
            )
            
            crew = Crew(agents=[analyst], tasks=[task], process=Process.sequential)
            raw_result = crew.kickoff()
            st.session_state["relatorio_analise"] = str(raw_result)

    # Exibição persistente e exportação
    if "relatorio_analise" in st.session_state:
        st.success("✅ Análise Concluída!")
        st.markdown("--- ")
        st.markdown(st.session_state["relatorio_analise"])
        
        st.download_button(
            label="📥 Baixar Dossiê Executivo (.md)",
            data=st.session_state["relatorio_analise"],
            file_name="dossie_executivo_bi.md",
            mime="text/markdown"
        )

elif not api_key:
    st.warning("⚠️ Por favor, insira sua API Key no menu lateral para começar.")

### 🌐 3. Execução & Túnel
Insira seu Token do Ngrok no campo abaixo e rode a célula para abrir o Dashboard.

In [ ]:
# @title Configuração do Túnel (Ngrok)
NGROK_TOKEN = "" # @param {type:"string"}

from pyngrok import ngrok
import os

if NGROK_TOKEN:
    # Configura o Token e abre o túnel na porta padrão do Streamlit (8501)
    ngrok.kill() # Fecha túneis antigos para evitar erros de limite de sessão
    ngrok.set_auth_token(NGROK_TOKEN)
    
    print("⌛ Aguarde, estabelecendo conexão segura...")
    public_url = ngrok.connect(8501).public_url
    print(f"🔗 DASHBOARD ATIVO EM: {public_url}")
    print("--- ")
    
    # Dispara o Streamlit
    !streamlit run app.py --server.port 8501
else:
    print("❌ ERRO: Você precisa inserir o NGROK_TOKEN para gerar o link de acesso.")